In [1]:
import os
import sys
import torch
from dataclasses import dataclass
from typing import Dict
from pathlib import Path

@dataclass
class Config:
    # Parâmetros de dados
    MIN_VOLUME = 2000
    TOP_LEVELS = 50
    
    # Thresholds e pesos para cálculos
    DISTANCE_THRESHOLD = 0.001
    CONFLUENCE_THRESHOLD = 0.002
    
    # Pesos para cálculo de força dos níveis
    VOLUME_WEIGHT = 0.35        # Ajustado de 0.4
    PRICE_ACTION_WEIGHT = 0.30  # Mantido em 0.3
    RETEST_WEIGHT = 0.20       # Mantido em 0.2
    CONFLUENCE_WEIGHT = 0.15    # Ajustado de 0.1
    
    # Diretórios
    BASE_DIR = "C:/ARQUIVOS/OneDrive/Bolsa/COLETA/TV/FUTUROS"
    CACHE_DIR = os.path.join("C:/ARQUIVOS/OneDrive/Bolsa/COLETA/TV/FUTUROS/cache")
    MODEL_SAVE_PATH = os.path.join(CACHE_DIR, "models")
    LOG_PATH = os.path.join(CACHE_DIR, "logs")
    
    # Arquivos
    REPORT_FILE = "market_report.txt"
    LOG_FILE = "market_analyzer.log"

    # Parâmetros técnicos
    RSI_OVERBOUGHT = 70
    RSI_OVERSOLD = 30
    STOCH_THRESHOLD = 80
    VOLUME_AVG_PERIODS = 20
    
    # Parâmetros de análise
    MIN_TOUCHES = 2
    MAX_GAP_SIZE = 0.01  # 1% para identificação de gaps
    MIN_LEVEL_STRENGTH = 0.3

    # Atualizar o TV_COLUMNS na classe Config
    TV_COLUMNS = {
        'time': 'DATA',
        'open': 'PREÇO ABERT.',
        'high': 'PREÇO MÁX.',
        'low': 'PREÇO MÍN.',
        'close': 'ÚLT. PREÇO',
        'VWAP': 'VWAP',
        'BUY+SELL V': 'VOLUME',
        'Total Buy Volume': 'VOLUME COMPRA',
        'Total Sell Volume': 'VOLUME VENDA',
        'Total Buy/Sell Ratio': 'BUY/SELL RATIO',
        'RSI': 'RSI',
        'Regular Bullish': 'BULLISH',
        'Regular Bearish': 'BEARISH',
        'EMA': 'EMA',
        'MA': 'SMA',
        '%K': 'STOCH_K',  # Corrigido
        '%D': 'STOCH_D',  # Corrigido
        'Upper Band #1': 'BB_UPPER',
        'Lower Band #1': 'BB_LOWER',
        'pdPOC': 'POC_D',
        'pdVAH': 'VAH_D',
        'pdVAL': 'VAL_D',
        'pmPOC': 'POC_M',
        'pmVAH': 'VAH_M',
        'pmVAL': 'VAL_M',
        'pwPOC': 'POC_W',
        'pwVAH': 'VAH_W',
        'pwVAL': 'VAL_W'
    }
    
    # Adicionar constantes para indicadores
    REQUIRED_INDICATORS = [
        'STOCH_K', 
        'STOCH_D', 
        'RSI', 
        'EMA', 
        'SMA', 
        'MACD',
        'ATR'
    ]

    # Mapeamento de colunas limpas (para processamento)
    CLEAN_COLUMNS = {
        'time': 'DATA',
        'open': 'PRECO_ABERTURA',
        'high': 'PRECO_MAXIMO',
        'low': 'PRECO_MINIMO',
        'close': 'ULTIMO_PRECO',
        'VWAP': 'VWAP',
        'BUY+SELL V': 'VOLUME',
        'Total Buy Volume': 'VOLUME_COMPRA',
        'Total Sell Volume': 'VOLUME_VENDA',
        'Total Buy/Sell Ratio': 'BUY_SELL_RATIO',
        'RSI': 'RSI',
        'Regular Bullish': 'BULLISH',
        'Regular Bearish': 'BEARISH',
        'EMA': 'EMA',
        'MA': 'SMA',
        '%K': 'STOCH_K',
        '%D': 'STOCH_D',
        'Upper Band #1': 'BB_UPPER',
        'Lower Band #1': 'BB_LOWER',
        'pdPOC': 'POC_D',
        'pdVAH': 'VAH_D',
        'pdVAL': 'VAL_D',
        'pmPOC': 'POC_M',
        'pmVAH': 'VAH_M',
        'pmVAL': 'VAL_M',
        'pwPOC': 'POC_W',
        'pwVAH': 'VAH_W',
        'pwVAL': 'VAL_W'
    }

    # Parâmetros do modelo Deep Learning
    max_prediction_length = 24
    max_encoder_length = 60
    batch_size = 32
    num_workers = 0
    learning_rate = 0.03
    hidden_size = 32
    attention_head_size = 2
    dropout = 0.1
    hidden_continuous_size = 16
    max_epochs = 30
    
    # Features para o modelo
    time_varying_known_reals = [
        'VWAP', 'RSI', 'STOCH_K', 'STOCH_D',
        'VOLUME', 'VOLUME_COMPRA', 'VOLUME_VENDA'
    ]
    
    time_varying_unknown_reals = [
        'ULTIMO_PRECO', 'PRECO_MAXIMO', 'PRECO_MINIMO',
        'POC_D', 'VAH_D', 'VAL_D'
    ]

    # Configurações de hardware
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
    def __post_init__(self):
        """Executado após a inicialização da classe"""
        # Cria diretórios necessários
        for path in [self.CACHE_DIR, self.MODEL_SAVE_PATH, self.LOG_PATH]:
            Path(path).mkdir(parents=True, exist_ok=True)
        
        # Valida configurações
        self._validate_config()
    
    def _validate_config(self):
        """Valida as configurações"""
        # Verifica diretório base
        if not os.path.exists(self.BASE_DIR):
            raise ValueError(f"Diretório base não encontrado: {self.BASE_DIR}")
        
        # Verifica pesos
        weights = [self.VOLUME_WEIGHT, self.PRICE_ACTION_WEIGHT, 
                  self.RETEST_WEIGHT, self.CONFLUENCE_WEIGHT]
        total_weight = sum(weights)
        if abs(total_weight - 1.0) > 1e-10:  # Tolerância para erros de arredondamento
            raise ValueError(f"Soma dos pesos deve ser 1.0 (atual: {total_weight:.10f})")
        
        # Verifica parâmetros do modelo
        if self.batch_size <= 0:
            raise ValueError("batch_size deve ser positivo")
        if self.max_epochs <= 0:
            raise ValueError("max_epochs deve ser positivo")
            
    def get_timeframe_config(self, timeframe: str) -> Dict:
        """Retorna configurações específicas para cada timeframe"""
        configs = {
            'D': {'min_touches': 2, 'volume_threshold': self.MIN_VOLUME},
            'W': {'min_touches': 3, 'volume_threshold': self.MIN_VOLUME * 5},
            'M': {'min_touches': 4, 'volume_threshold': self.MIN_VOLUME * 20}
        }
        return configs.get(timeframe, configs['D'])

    def get_column_mapping(self, clean: bool = False) -> Dict:
        """Retorna mapeamento de colunas apropriado"""
        return self.CLEAN_COLUMNS if clean else self.TV_COLUMNS

In [2]:
import pandas as pd
import numpy as np
import logging
from typing import Tuple, List, Dict, Optional
from datetime import datetime
import os
from pathlib import Path

class DataProcessor:
    def __init__(self, config: Config):
        self.config = config
        self.setup_logging()
        self.validate_column_mapping()  # Adicionar esta linha
        
    def setup_logging(self):
        """Configura sistema de logging"""
        log_file = Path(self.config.LOG_PATH) / "data_processor.log"
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    
    def load_and_process_data(self, filepath: str) -> Optional[pd.DataFrame]:
        try:
            self.logger.info(f"Carregando arquivo: {filepath}")
            # Carregar o CSV com parse de datas
            df = pd.read_csv(filepath)

            # Check if 'time' column contains Unix timestamps
            try:
                 # Attempt to convert 'time' column assuming Unix timestamps in seconds
                 df['DATA'] = pd.to_datetime(df['time'], unit='s')
            except (ValueError, TypeError):
                 try:
                   # If that fails, try milliseconds
                   df['DATA'] = pd.to_datetime(df['time'], unit='ms')
                 except (ValueError, TypeError):
                       try:
                          # If both fail, try parsing as string
                          df['DATA'] = pd.to_datetime(df['time'])
                       except (ValueError, TypeError):
                         self.logger.error(f"Não foi possível converter a coluna 'time' para datetime")
                         raise
            
            # Set timezone to UTC and convert to São Paulo time
            df['DATA'] = df['DATA'].dt.tz_localize('UTC').dt.tz_convert('America/Sao_Paulo')
            
            # Drop original time column
            df = df.drop('time', axis=1)
            
            self._inspect_data(df, "Dados originais")
            df = self._preprocess_data(df)
            self._validate_data(df)
            self._inspect_data(df, "Dados processados")
            
            return df
            
        except Exception as e:
            self.logger.error(f"Erro no processamento de dados: {str(e)}")
            raise

    def _normalize_column_name(self, col: str) -> str:
        """Normaliza nomes das colunas tratando caracteres especiais"""
        char_map = {
            'М': 'M',  # Cirílico para latino
            'á': 'a', 'í': 'i', 'ó': 'o', 'ú': 'u', 'é': 'e',
            'ã': 'a', 'õ': 'o', 'â': 'a', 'ê': 'e', 'î': 'i',
            'ô': 'o', 'û': 'u', 'х': 'x', 'ç': 'c',
            'Á': 'A', 'Í': 'I', 'Ó': 'O', 'Ú': 'U', 'É': 'E',
            'Ã': 'A', 'Õ': 'O', 'Â': 'A', 'Ê': 'E', 'Î': 'I',
            'Ô': 'O', 'Û': 'U', 'Х': 'X', 'Ç': 'C'
        }
        
        # Pre-processamento para substituir caracteres cirílicos
        for old_char, new_char in char_map.items():
            col = col.replace(old_char, new_char)
            
        # Substitui caracteres especiais
        col = ''.join(c if c.isalnum() or c in ['_', ' '] else '_' for c in col)
        return col.replace(' ', '_').upper()

    
    def _inspect_data(self, df: pd.DataFrame, stage: str):
        """Inspeção detalhada dos dados com logging aprimorado"""
        try:
            self.logger.info(f"\n{stage}:")
            self.logger.info(f"Dimensões: {df.shape}")
            self.logger.info("Tipos de dados:")
            for col, dtype in df.dtypes.items():
                self.logger.info(f"  {col}: {dtype}".encode('utf-8'))
                
            self.logger.info("\nValores nulos por coluna:")
            null_counts = df.isnull().sum()
            for col, count in null_counts.items():
                if count > 0:
                    self.logger.info(f"  {col}: {count} ({count/len(df)*100:.1f}%)")
                    
            # Log column changes during processing
            if stage == "Dados processados":
                original_cols = set(df.columns)
                expected_cols = set(self.config.CLEAN_COLUMNS.values())
                missing_cols = expected_cols - original_cols
                extra_cols = original_cols - expected_cols
                
                if missing_cols:
                    self.logger.warning(f"Colunas esperadas ausentes: {missing_cols}")
                if extra_cols:
                    self.logger.info(f"Colunas adicionais encontradas: {extra_cols}")
                    
        except Exception as e:
            self.logger.error(f"Erro na inspeção de dados: {str(e)}")
            raise

    def _preprocess_data(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        
        if 'DATA' in df.columns:
            # Check for invalid dates
            invalid_dates = df[df['DATA'].dt.year > datetime.now().year + 1]
            if not invalid_dates.empty:
                self.logger.warning(f"Encontradas {len(invalid_dates)} datas inválidas")
                # Remove invalid dates
                df = df[df['DATA'].dt.year <= datetime.now().year + 1]
            
            # Ensure dates are in chronological order
            df = df.sort_values('DATA')
        
        # Limpa nomes das colunas
        df = self._clean_column_names(df)
        
        # Verifica colunas necessárias
        required_columns = ['RSI', 'EMA', 'SMA', 'VWAP', 'STOCH_K', 'STOCH_D', 'ULTIMO_PRECO', 'VOLUME']
        missing_columns = [col for col in required_columns if col not in df.columns]
        if missing_columns:
            self.logger.error(f"Colunas necessárias ausentes: {missing_columns}")
            raise ValueError(f"Colunas necessárias ausentes: {missing_columns}")
        
        # Adiciona colunas necessárias
        df = self._add_required_columns(df)
        
        # Trata valores ausentes
        df = self._handle_missing_values(df)
        
        # Calcula campos derivados
        df = self._calculate_derived_fields(df)
        
        return df
        
    def _clean_column_names(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        column_mapping = {}
        used_names = set()
    
        for col in df.columns:
            if col in self.config.CLEAN_COLUMNS:
                new_name = self.config.CLEAN_COLUMNS[col]
            else:
                new_name = self._normalize_column_name(col)
            
            if new_name in used_names:
                counter = 1
                while f"{new_name}_{counter}" in used_names:
                    counter += 1
                new_name = f"{new_name}_{counter}"
            
            column_mapping[col] = new_name
            used_names.add(new_name)
    
        df = df.rename(columns=column_mapping)
        self.logger.debug(f"Mapeamento final de colunas: {column_mapping}")
        return df
    
    def validate_column_mapping(self):
        """Validações para o mapeamento de colunas"""
        try:
            # Verifica duplicatas nos valores dos mapeamentos
            tv_values = list(self.config.TV_COLUMNS.values())
            clean_values = list(self.config.CLEAN_COLUMNS.values())
            
            # Usa sets para detectar duplicatas
            tv_duplicates = set([x for x in tv_values if tv_values.count(x) > 1])
            clean_duplicates = set([x for x in clean_values if clean_values.count(x) > 1])
            
            if tv_duplicates:
                self.logger.warning(f"Duplicatas encontradas no TV_COLUMNS: {tv_duplicates}")
            if clean_duplicates:
                self.logger.warning(f"Duplicatas encontradas no CLEAN_COLUMNS: {clean_duplicates}")
                
            # Verifica consistência entre os mapeamentos
            for tv_col in self.config.TV_COLUMNS.values():
                if tv_col not in self.config.CLEAN_COLUMNS:
                    self.logger.debug(f"Coluna TV sem mapeamento CLEAN: {tv_col}")
                    
            return True
            
        except Exception as e:
            self.logger.error(f"Erro na validação do mapeamento: {str(e)}")
            raise
    
    def _add_required_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adiciona colunas necessárias para análise"""
        # Índice temporal
        if 'time_idx' not in df.columns:
            df['time_idx'] = range(len(df))
            
        # Identificador de grupo
        if 'group' not in df.columns:
            df['group'] = 'WDO'
            
        # Volume normalizado
        if 'VOLUME' in df.columns:
            df['VOLUME_NORMALIZADO'] = df['VOLUME'] / df['VOLUME'].rolling(
                window=self.config.VOLUME_AVG_PERIODS).mean()
            
        # Variação percentual
        if 'ULTIMO_PRECO' in df.columns:
            df['VAR_PCT'] = df['ULTIMO_PRECO'].pct_change() * 100
            
        return df

    def _handle_missing_values(self, df: pd.DataFrame) -> pd.DataFrame:
        """Tratamento abrangente de valores ausentes"""
        # Identificação de colunas por tipo
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        categorical_cols = ['BULLISH', 'BEARISH']
        indicator_cols = ['RSI', 'STOCH_K', 'STOCH_D', 'EMA', 'SMA']
        price_cols = ['ULTIMO_PRECO', 'PRECO_MAXIMO', 'PRECO_MINIMO', 'PRECO_ABERTURA']
        
        # Tratamento por tipo de coluna
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
        df[categorical_cols] = df[categorical_cols].fillna(False)
        df[indicator_cols] = df[indicator_cols].ffill()
        df[price_cols] = df[price_cols].ffill()
        
        # Tratamento específico para BULLISH e BEARISH
        if 'BULLISH' in df.columns and 'BEARISH' in df.columns:
            df['BULLISH'] = df['BULLISH'].fillna(0).astype(bool)
            df['BEARISH'] = df['BEARISH'].fillna(0).astype(bool)
        else:
            self.logger.warning("Colunas BULLISH e/ou BEARISH não encontradas. Criando colunas vazias.")
            df['BULLISH'] = False
            df['BEARISH'] = False

        if 'FORCA_COMPRADORA' in df.columns:
            df['FORCA_COMPRADORA'] = df['FORCA_COMPRADORA'].fillna(0)
            
        return df

    def _calculate_derived_fields(self, df: pd.DataFrame) -> pd.DataFrame:
        """Calcula campos derivados para análise"""
        try:
            # Médias móveis
            if 'ULTIMO_PRECO' in df.columns:
                df['SMA_20'] = df['ULTIMO_PRECO'].rolling(window=20).mean()
                df['SMA_50'] = df['ULTIMO_PRECO'].rolling(window=50).mean()
                
            # Volatilidade
            if all(col in df.columns for col in ['PRECO_MAXIMO', 'PRECO_MINIMO', 'ULTIMO_PRECO']):
                df['VOLATILIDADE'] = ((df['PRECO_MAXIMO'] - df['PRECO_MINIMO']) / 
                                    df['ULTIMO_PRECO'] * 100)
                
            # Volume médio
            if 'VOLUME' in df.columns:
                df['VOLUME_MEDIO'] = df['VOLUME'].rolling(
                    window=self.config.VOLUME_AVG_PERIODS).mean()
                
            # Força compradora/vendedora
            if all(col in df.columns for col in ['VOLUME_COMPRA', 'VOLUME_VENDA']):
                df['FORCA_COMPRADORA'] = (df['VOLUME_COMPRA'] / 
                                        (df['VOLUME_COMPRA'] + df['VOLUME_VENDA']))
                
            return df
            
        except Exception as e:
            self.logger.error(f"Erro ao calcular campos derivados: {str(e)}")
            raise

    def _validate_data(self, df: pd.DataFrame):
        """Validação completa dos dados processados"""
        # Verifica colunas obrigatórias
        required_cols = (
            ['DATA', 'ULTIMO_PRECO', 'VOLUME'] +
            self.config.time_varying_known_reals +
            self.config.time_varying_unknown_reals
        )
        
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Colunas obrigatórias ausentes: {missing_cols}")
            
        # Verifica tipos de dados
        if not pd.api.types.is_datetime64_any_dtype(df['DATA']):
            raise ValueError("Coluna 'DATA' deve ser do tipo datetime")
            
        numeric_cols = ['ULTIMO_PRECO', 'VOLUME']
        for col in numeric_cols:
            if not pd.api.types.is_numeric_dtype(df[col]):
                raise ValueError(f"Coluna '{col}' deve ser numérica")
                
        # Verifica ordenação temporal
        if not df['DATA'].is_monotonic_increasing:
            raise ValueError("Dados devem estar ordenados cronologicamente")
            
        # Verifica valores negativos em colunas que não podem ser negativas
        non_negative_cols = ['VOLUME', 'VOLUME_COMPRA', 'VOLUME_VENDA']
        for col in non_negative_cols:
            if col in df.columns and (df[col] < 0).any():
                raise ValueError(f"Coluna '{col}' contém valores negativos")

        # Verifica presença das colunas BULLISH e BEARISH
        if 'BULLISH' not in df.columns or 'BEARISH' not in df.columns:
            raise ValueError("Colunas BULLISH e BEARISH são obrigatórias")
        
        # Verifica se são do tipo bool
        if not pd.api.types.is_bool_dtype(df['BULLISH']) or not pd.api.types.is_bool_dtype(df['BEARISH']):
            raise ValueError("Colunas BULLISH e BEARISH devem ser do tipo booleano")


    def prepare_train_val_data(self, df: pd.DataFrame, 
                             val_size: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Prepara dados de treino e validação"""
        # Garante que os dados estão ordenados
        df = df.sort_values('DATA').reset_index(drop=True)
        
        # Calcula ponto de corte
        cutoff_idx = int(len(df) * (1 - val_size))
        
        # Separa dados
        train_df = df.iloc[:cutoff_idx].copy()
        val_df = df.iloc[cutoff_idx:].copy()
        
        self.logger.info(f"Dados separados em treino ({len(train_df)} registros) "
                        f"e validação ({len(val_df)} registros)")
        
        return train_df, val_df

    def save_processed_data(self, df: pd.DataFrame, timeframe: str):
        """Salva dados processados"""
        output_path = Path(self.config.CACHE_DIR) / f"processed_{timeframe}.parquet"
        df.to_parquet(output_path)
        self.logger.info(f"Dados processados salvos em {output_path}")

    def load_processed_data(self, timeframe: str) -> Optional[pd.DataFrame]:
        """Carrega dados processados"""
        input_path = Path(self.config.CACHE_DIR) / f"processed_{timeframe}.parquet"
        if input_path.exists():
            return pd.read_parquet(input_path)
        return None

In [3]:
import pandas as pd
import numpy as np
import logging
from typing import List, Dict, Tuple, Optional
from datetime import datetime
from pathlib import Path
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from .tft_model import TFTModel
    
class MarketAnalyzer:
    def __init__(self, config: Config):
        self.config = config
        self.setup_logging()
        
    def setup_logging(self):
        log_file = Path(self.config.LOG_PATH) / f"system_{datetime.now():%Y%m%d}.log"
        logging.basicConfig(
            level=logging.WARNING,  # Ajuste o nível de log aqui
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)


    def identify_levels_with_tft(self, df: pd.DataFrame, tft_model: 'TFTModel') -> Tuple[List[Dict], List[Dict]]:
        """
        Identifica níveis de suporte e resistência incorporando previsões do modelo TFT.
        
        Args:
            df (pd.DataFrame): DataFrame com dados históricos
            tft_model (TFTModel): Modelo TFT treinado
        
        Returns:
            Tuple[List[Dict], List[Dict]]: Listas de suportes e resistências identificados
        """
        try:
            # Identificação inicial de níveis usando o método existente
            initial_supports, initial_resistances = self.identify_levels(df)
            
            # Obter previsões do modelo TFT
            predictions = tft_model.predict(df)
            forecast = predictions['forecast'].squeeze()  # Assumindo que é uma série de previsões
            lower_bound = predictions['lower_bound'].squeeze()
            upper_bound = predictions['upper_bound'].squeeze()
            
            last_price = df['ULTIMO_PRECO'].iloc[-1]
            future_prices = forecast.tolist()
            
            # Função para ajustar a força do nível com base nas previsões
            def adjust_level_strength(level: Dict, future_prices: List[float]) -> float:
                price = level['price']
                original_strength = level['strength']
                future_touches = sum(1 for fp in future_prices if abs(fp - price) / price < self.config.DISTANCE_THRESHOLD)
                strength_boost = future_touches / len(future_prices)
                return min(original_strength + strength_boost, 1.0)
            
            # Ajustar força dos níveis existentes
            for level in initial_supports + initial_resistances:
                level['strength'] = adjust_level_strength(level, future_prices)
            
            # Identificar novos níveis potenciais das previsões
            potential_levels = self._find_potential_levels(future_prices, lower_bound, upper_bound)
            
            # Adicionar novos níveis potenciais à lista existente
            all_levels = initial_supports + initial_resistances + potential_levels
            
            # Recalcular força e ordenar níveis
            for level in all_levels:
                level['strength'] = self._calculate_level_strength(df, level)
            
            all_levels.sort(key=lambda x: x['strength'], reverse=True)
            
            # Separar em suportes e resistências
            supports = [level for level in all_levels if level['price'] < last_price]
            resistances = [level for level in all_levels if level['price'] >= last_price]
            
            return supports[:self.config.TOP_LEVELS], resistances[:self.config.TOP_LEVELS]
        
        except Exception as e:
            self.logger.error(f"Erro na identificação de níveis com TFT: {str(e)}")
            raise
    
    def _find_potential_levels(self, future_prices: List[float], lower_bound: List[float], upper_bound: List[float]) -> List[Dict]:
        """
        Identifica níveis potenciais a partir das previsões do modelo TFT.
        """
        potential_levels = []
        for i, price in enumerate(future_prices):
            if i > 0 and i < len(future_prices) - 1:
                if (future_prices[i-1] < price > future_prices[i+1]) or (future_prices[i-1] > price < future_prices[i+1]):
                    strength = (upper_bound[i] - lower_bound[i]) / price  # Menor intervalo indica maior confiança
                    potential_levels.append({
                        'price': price,
                        'strength': 1 - strength,  # Inverte para que menor intervalo resulte em maior força
                        'type': 'tft_prediction',
                        'volume': 0,  # Não temos volume para previsões futuras
                        'touches': 1  # Consideramos como um toque inicial
                    })
        
        return potential_levels
    
    def analyze_market_context(self, df: pd.DataFrame) -> Dict:
        """Análise completa do contexto de mercado"""
        try:
            last_row = df.iloc[-1]
            last_price = last_row['ULTIMO_PRECO']
            vwap = last_row['VWAP']
            
            # Determina tendência
            trend = self._determine_trend(df)
            
            # Calcula volatilidade  
            volatility = self._calculate_volatility(df)
            
            # Análise de volume
            volume_analysis = self._analyze_volume(df)
            
            # Força compradora/vendedora
            buy_sell_analysis = self._analyze_buy_sell_strength(df)
            
            # Análise técnica
            technical_analysis = self._analyze_technical_indicators(df)
            
            # Determina força da tendência
            market_condition = self._assess_market_condition(df)
            
            # Obtém áreas de valor
            value_areas = self._get_value_areas(df)
    
            # Obtém níveis de suporte e resistência
            supports, resistances = self.identify_levels(df)
            
            # Encontra próximos níveis
            next_resistance = float('inf')
            next_support = 0
            resistance_distance = 0
            support_distance = 0
            
            # Encontra próxima resistência (menor preço acima do atual)
            for r in sorted(resistances, key=lambda x: x['price']):
                if r['price'] > last_price:
                    next_resistance = r['price']
                    resistance_distance = ((next_resistance - last_price) / last_price) * 100
                    break
                    
            # Encontra próximo suporte (maior preço abaixo do atual)
            for s in sorted(supports, key=lambda x: x['price'], reverse=True):
                if s['price'] < last_price:
                    next_support = s['price']
                    support_distance = ((last_price - next_support) / last_price) * 100
                    break
    
            # Obtém indicadores técnicos do último registro
            rsi = technical_analysis['rsi']['value']
            stoch_k = technical_analysis['stochastic']['k']
            stoch_d = technical_analysis['stochastic']['d']
            
            # Obtém sinais técnicos
            bullish_signals = "Sim" if last_row.get('BULLISH', False) else "Não"
            bearish_signals = "Sim" if last_row.get('BEARISH', False) else "Não"
            stoch_signal = technical_analysis['stochastic']['signal']
            rsi_signal = technical_analysis['rsi']['condition']
            ma_signal = technical_analysis['moving_averages']['signal']
    
            # Adiciona campos Bollinger
            bb_upper = technical_analysis['bollinger']['upper']
            bb_lower = technical_analysis['bollinger']['lower']
            
            return {
                'trend': trend,
                'trend_strength': market_condition['trend_strength'],
                'volatility': volatility,
                'last_price': last_price, 
                'vwap': vwap,
                'volume_analysis': volume_analysis,
                'avg_volume': volume_analysis['average'],
                'buy_sell_analysis': buy_sell_analysis,
                'buy_sell_ratio': buy_sell_analysis['ratio'],
                'technical_analysis': technical_analysis,
                'value_areas': value_areas,
                'market_condition': market_condition,
                'rsi': rsi,
                'stoch_k': stoch_k,
                'stoch_d': stoch_d,
                'poc_d': value_areas['daily']['poc'],
                'vah_d': value_areas['daily']['vah'],
                'val_d': value_areas['daily']['val'],
                'poc_w': value_areas['weekly']['poc'],
                'vah_w': value_areas['weekly']['vah'],
                'val_w': value_areas['weekly']['val'],
                'poc_m': value_areas['monthly']['poc'],
                'vah_m': value_areas['monthly']['vah'],
                'val_m': value_areas['monthly']['val'],
                'bb_upper': bb_upper,
                'bb_lower': bb_lower,
                'vwap_distance': ((last_price - vwap) / vwap * 100) if vwap != 0 else 0,
                'vwap_position': 'Acima' if last_price > vwap else 'Abaixo' if last_price < vwap else 'Na VWAP',
                # Adicionando próximos níveis e suas distâncias
                'next_resistance': next_resistance if next_resistance != float('inf') else last_price,
                'next_support': next_support if next_support != 0 else last_price,
                'resistance_distance': resistance_distance,
                'support_distance': support_distance,
                # Adicionando sinais técnicos
                'bullish_signals': bullish_signals,
                'bearish_signals': bearish_signals,
                'stoch_signal': stoch_signal,
                'rsi_signal': rsi_signal,
                'ma_signal': ma_signal
            }
            
        except Exception as e:
            self.logger.error(f"Erro na análise de contexto: {str(e)}")
            raise


    def identify_levels(self, df: pd.DataFrame) -> Tuple[List[Dict], List[Dict]]:
        """Identifica níveis de suporte e resistência"""
        try:
            last_close = df['ULTIMO_PRECO'].iloc[-1]
            supports = []
            resistances = []
            
            # Análise por volume
            volume_levels = self._find_volume_levels(df)
            
            # Níveis de referência (POC, VAH, VAL)
            reference_levels = self._get_reference_levels(df)
            
            # Combina e processa níveis
            all_levels = self._process_levels(df, volume_levels, reference_levels)
            
            # Separa suportes e resistências
            for level in all_levels:
                if level['price'] < last_close:
                    supports.append(level)
                else:
                    resistances.append(level)
            
            # Ordena por força
            supports = sorted(supports, key=lambda x: x['strength'], reverse=True)
            resistances = sorted(resistances, key=lambda x: x['strength'], reverse=True)
            
            return supports[:self.config.TOP_LEVELS], resistances[:self.config.TOP_LEVELS]
            
        except Exception as e:
            self.logger.error(f"Erro na identificação de níveis: {str(e)}")
            raise

    def _determine_trend(self, df: pd.DataFrame) -> str:
        """Determina a tendência do mercado"""
        last_price = df['ULTIMO_PRECO'].iloc[-1]
        vwap = df['VWAP'].iloc[-1]
        ema = df['EMA'].iloc[-1]
        sma = df['SMA'].iloc[-1]
        
        if last_price > vwap * 1.01 and ema > sma:
            return "ALTA ↑"
        elif last_price < vwap * 0.99 and ema < sma:
            return "BAIXA ↓"
        else:
            return "LATERAL →"

    def _calculate_volatility(self, df: pd.DataFrame) -> float:
        """Calcula a volatilidade do mercado"""
        return ((df['PRECO_MAXIMO'] - df['PRECO_MINIMO']) / 
                df['VWAP']).mean() * 100

    def _analyze_volume(self, df: pd.DataFrame) -> Dict:
        """Analisa perfil de volume"""
        recent_df = df.tail(self.config.VOLUME_AVG_PERIODS)
        avg_volume = recent_df['VOLUME'].mean()
        vol_trend = recent_df['VOLUME'].pct_change().mean()
        
        return {
            'average': avg_volume,
            'trend': vol_trend,
            'above_average': recent_df['VOLUME'].iloc[-1] > avg_volume,
            'distribution': self._analyze_volume_distribution(df)
        }

    def _analyze_buy_sell_strength(self, df: pd.DataFrame) -> Dict:
        """Analisa força compradora/vendedora"""
        recent_df = df.tail(5)
        buy_volume = recent_df['VOLUME_COMPRA'].sum()
        sell_volume = recent_df['VOLUME_VENDA'].sum()
        ratio = buy_volume / (buy_volume + sell_volume)
        
        return {
            'buy_volume': buy_volume,
            'sell_volume': sell_volume,
            'ratio': ratio,
            'pressure': 'Compradora ↑' if ratio > 0.5 else 'Vendedora ↓'
        }

    def _analyze_technical_indicators(self, df: pd.DataFrame) -> Dict:
        """Analisa indicadores técnicos"""
        last_row = df.iloc[-1]
        
        return {
            'rsi': {
                'value': last_row['RSI'],
                'condition': 'SOBRECOMPRADO' if last_row['RSI'] > 70 else 
                            'SOBREVENDIDO' if last_row['RSI'] < 30 else 'NEUTRO'
            },
            'stochastic': {
                'k': last_row['STOCH_K'],
                'd': last_row['STOCH_D'],
                'signal': 'BULLISH' if last_row['STOCH_K'] > last_row['STOCH_D'] else 'BEARISH'
            },
            'moving_averages': {
                'ema': last_row['EMA'],
                'sma': last_row['SMA'],
                'signal': 'BULLISH' if last_row['EMA'] > last_row['SMA'] else 'BEARISH'
            },
            'bollinger': {
                'upper': last_row['BB_UPPER'],
                'lower': last_row['BB_LOWER'],
                'bandwidth': (last_row['BB_UPPER'] - last_row['BB_LOWER']) / last_row['ULTIMO_PRECO']
            }
        }

    def _find_volume_levels(self, df: pd.DataFrame) -> Dict[float, Dict]:
        """Encontra níveis importantes baseados em volume"""
        levels = {}
        
        for _, row in df.iterrows():
            price = row['ULTIMO_PRECO']
            volume = row['VOLUME']
            
            # Agrupa volumes por níveis de preço próximos
            for level in levels:
                if abs(price - level) / level < self.config.DISTANCE_THRESHOLD:
                    levels[level]['volume'] += volume
                    levels[level]['touches'] += 1
                    levels[level]['last_touch'] = row['DATA']
                    break
            else:
                levels[price] = {
                    'volume': volume,
                    'touches': 1,
                    'first_touch': row['DATA'],
                    'last_touch': row['DATA']
                }
        
        return levels

    def _process_levels(self, df: pd.DataFrame, volume_levels: Dict, 
                       reference_levels: Dict) -> List[Dict]:
        """Processa e combina diferentes tipos de níveis"""
        processed_levels = []
        
        # Processa níveis de volume
        for price, data in volume_levels.items():
            if data['volume'] >= self.config.MIN_VOLUME:
                # Criar o dicionário level com os dados necessários
                level = {
                    'price': price,
                    'volume': data['volume'],
                    'touches': data['touches'],
                    'first_touch': data['first_touch'],
                    'last_touch': data['last_touch']
                }
                
                strength = self._calculate_level_strength(df, level)
                processed_levels.append({
                    'price': price,
                    'volume': data['volume'],
                    'touches': data['touches'],
                    'strength': strength,
                    'first_touch': data['first_touch'],
                    'last_touch': data['last_touch'],
                    'type': 'volume'
                })
        
        # Adiciona níveis de referência
        for name, price in reference_levels.items():
            if pd.notna(price):
                # Criar o dicionário level para níveis de referência
                level = {
                    'price': price,
                    'volume': 0,
                    'touches': 0
                }
                strength = self._calculate_level_strength(df, level)
                processed_levels.append({
                    'price': price,
                    'volume': 0,
                    'touches': 0,
                    'strength': strength,
                    'type': name
                })
        
        return processed_levels


    def _calculate_level_strength(self, df: pd.DataFrame, level: Dict) -> float:
        """Calcula força de um nível baseado em volume, toques, retestes e confluência
        
        Args:
            df: DataFrame com dados históricos
            level: Dicionário com informações do nível
            
        Returns:
            float: Valor de força entre 0 e 1
        """
        try:
            # Verifica campos necessários no dicionário level
            price = level['price']
            volume = level.get('volume', 0)
            touches = level.get('touches', 0)
            
            # Calcula componentes individuais
            volume_strength = self._calculate_volume_strength(volume) 
            touch_strength = self._calculate_touch_strength(touches)
            
            # Calcula força de reteste se houver data do último toque
            if 'last_touch' in level:
                # Converter last_touch para timezone-aware se necessário
                last_touch = pd.to_datetime(level['last_touch'])
                if last_touch.tz is None:
                    last_touch = last_touch.tz_localize('America/Sao_Paulo')
                
                days_since_touch = (pd.Timestamp.now(tz='America/Sao_Paulo') - last_touch).days
                retest_strength = max(1.0 - (days_since_touch / 30), 0.0)
            else:
                retest_strength = 0.0
                
            # Calcula força de confluência
            confluence_strength = self._calculate_confluence_strength(df, price)
            
            # Calcula força total ponderada
            total_strength = (
                volume_strength * self.config.VOLUME_WEIGHT +
                touch_strength * self.config.PRICE_ACTION_WEIGHT +
                retest_strength * self.config.RETEST_WEIGHT +
                confluence_strength * self.config.CONFLUENCE_WEIGHT
            )
            
            return min(total_strength, 1.0)
            
        except Exception as e:
            self.logger.error(f"Erro ao calcular força do nível: {str(e)}")
            return 0.0
    
    
    def _calculate_volume_strength(self, volume: float) -> float:
        """Calcula componente de força baseado em volume"""
        if volume == 0:
            return 0
        return min(volume / self.config.MIN_VOLUME, 1.0)

    def _calculate_touch_strength(self, touches: int) -> float:
        """Calcula componente de força baseado em toques"""
        return min(touches / 10, 1.0)

    def _calculate_retest_strength(self, last_touch: datetime) -> float:
        """Calcula componente de força baseado em retestes recentes"""
        days_since_touch = (datetime.now() - pd.to_datetime(last_touch)).days
        return max(1.0 - (days_since_touch / 30), 0.0)

    def _calculate_confluence_strength(self, df: pd.DataFrame, price: float) -> float:
        """Calcula componente de força baseado em confluência"""
        confluence = 0.0
        reference_levels = self._get_reference_levels(df)
        
        for ref_price in reference_levels.values():
            if pd.notna(ref_price):
                if abs(price - ref_price) / price < self.config.CONFLUENCE_THRESHOLD:
                    confluence += 0.2
                    
        return min(confluence, 1.0)

    def _get_reference_levels(self, df: pd.DataFrame) -> Dict[str, float]:
        """Obtém níveis de referência do último período"""
        last_row = df.iloc[-1]
        return {
            'POC_D': last_row.get('POC_D', np.nan),
            'VAH_D': last_row.get('VAH_D', np.nan),
            'VAL_D': last_row.get('VAL_D', np.nan),
            'POC_W': last_row.get('POC_W', np.nan),
            'VAH_W': last_row.get('VAH_W', np.nan),
            'VAL_W': last_row.get('VAL_W', np.nan),
            'POC_M': last_row.get('POC_M', np.nan),
            'VAH_M': last_row.get('VAH_M', np.nan),
            'VAL_M': last_row.get('VAL_M', np.nan)
        }

    def _get_value_areas(self, df: pd.DataFrame) -> Dict[str, Dict[str, float]]:
        """Obtém áreas de valor para diferentes timeframes"""
        last_row = df.iloc[-1]
        
        return {
            'daily': {
                'poc': last_row['POC_D'],
                'vah': last_row['VAH_D'],
                'val': last_row['VAL_D']
            },
            'weekly': {
                'poc': last_row['POC_W'],
                'vah': last_row['VAH_W'],
                'val': last_row['VAL_W']
            },
            'monthly': {
                'poc': last_row['POC_M'],
                'vah': last_row['VAH_M'],
                'val': last_row['VAL_M']
            }
        }

    def _assess_market_condition(self, df: pd.DataFrame) -> Dict:
        """Avalia condição geral do mercado"""
        trend = self._determine_trend(df)
        volatility = self._calculate_volatility(df)
        volume_analysis = self._analyze_volume(df)
        
        # Avalia força da tendência
        trend_strength = 'FORTE'
        if trend == 'LATERAL →':
            trend_strength = 'FRACA'
        elif abs(df['VAR_PCT'].mean()) < 0.5:
            trend_strength = 'MODERADA'
            
        return {
            'trend': trend,
            'trend_strength': trend_strength,
            'volatility_level': 'ALTA' if volatility > 2.0 else 'MÉDIA' if volatility > 1.0 else 'BAIXA',
            'volume_condition': 'ALTO' if volume_analysis['above_average'] else 'BAIXO'
        }

    def _analyze_volume_distribution(self, df: pd.DataFrame) -> Dict:
        """Analisa distribuição de volume"""
        volume_at_price = df.groupby('ULTIMO_PRECO')['VOLUME'].sum()
        total_volume = volume_at_price.sum()
        
        return {
            'concentration': volume_at_price.max() / total_volume,
            'distribution': volume_at_price.to_dict()
        }

In [4]:
import sys
import logging
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss, SMAPE, MAE, RMSE
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
import mlflow
import torch
from datetime import datetime
from pathlib import Path
from typing import Dict, Optional

class TFTModel(pl.LightningModule):
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self._setup_logging()
        self.setup_metrics()
        self.model = None
        self.training_cutoff = None
        self.save_hyperparameters(ignore=['loss'])
        
        # Verificação de GPU movida para dentro do __init__
        self._check_gpu()
        
    @property
    def custom_logger(self):
        """Retorna o logger da instância"""
        return self._logger

    def _check_gpu(self):
        """Verifica disponibilidade de GPU e memória"""
        if torch.cuda.is_available():
            self.custom_logger.info(f"GPU disponível: {torch.cuda.get_device_name(0)}")
            # Verifica memória GPU
            gpu_memory = torch.cuda.get_device_properties(0).total_memory
            if gpu_memory < 4e9:  # 4GB
                self.custom_logger.warning("GPU com pouca memória, considere reduzir batch_size")
        else:
            self.custom_logger.warning("GPU não disponível, usando CPU")
            
    def setup_metrics(self):
        """Configura métricas de avaliação"""
        self.metrics = {
            'smape': SMAPE(),
            'mae': MAE(),
            'rmse': RMSE(),
            'quantile_loss': QuantileLoss()
        }
    
    def _setup_logging(self):
        """Configura sistema de logging com suporte a UTF-8"""
        log_file = Path(self.config.LOG_PATH) / f"tft_model_{datetime.now():%Y%m%d}.log"
        
        # Força encoding UTF-8 no logging
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        
        file_handler = logging.FileHandler(log_file, encoding='utf-8')
        file_handler.setFormatter(formatter)
        
        stream_handler = logging.StreamHandler(sys.stdout)  # Usa stdout ao invés de stderr
        stream_handler.setFormatter(formatter)
        
        # Configura logger específico para esta instância
        self._logger = logging.getLogger(f"{__name__}.{id(self)}")
        self._logger.setLevel(logging.INFO)
        
        # Remove handlers existentes para evitar duplicação
        self._logger.handlers.clear()
        
        # Adiciona os novos handlers
        self._logger.addHandler(file_handler)
        self._logger.addHandler(stream_handler)

   
    def create_datasets(self, df: pd.DataFrame) -> Tuple[TimeSeriesDataSet, TimeSeriesDataSet]:
        """Cria datasets de treino e validação"""
        try:
            # Calcula ponto de corte para validação
            self.training_cutoff = df["time_idx"].max() - self.config.max_prediction_length
            
            # Cria dataset de treino
            training = TimeSeriesDataSet(
                df[lambda x: x.time_idx <= self.training_cutoff],
                time_idx="time_idx",
                target="ULTIMO_PRECO",
                group_ids=["group"],
                min_encoder_length=self.config.max_encoder_length // 2,
                max_encoder_length=self.config.max_encoder_length,
                min_prediction_length=1,
                max_prediction_length=self.config.max_prediction_length,
                static_categoricals=[],
                static_reals=[],
                time_varying_known_reals=self.config.time_varying_known_reals,
                time_varying_unknown_reals=self.config.time_varying_unknown_reals,
                target_normalizer=GroupNormalizer(
                    groups=["group"],
                    transformation="softplus"
                ),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
            )
            
            # Cria dataset de validação
            validation = TimeSeriesDataSet.from_dataset(training, df, predict=True)
            
            return training, validation
            
        except Exception as e:
            self.custom_logger.error(f"Erro ao criar datasets: {str(e)}")
            raise

    def create_dataloaders(self, training: TimeSeriesDataSet, 
                          validation: TimeSeriesDataSet) -> Tuple[torch.utils.data.DataLoader, 
                                                                torch.utils.data.DataLoader]:
        """Cria dataloaders para treino e validação"""
        try:
            train_dataloader = training.to_dataloader(
                train=True,
                batch_size=self.config.batch_size,
                num_workers=self.config.num_workers
            )
            
            val_dataloader = validation.to_dataloader(
                train=False,
                batch_size=self.config.batch_size * 2,
                num_workers=self.config.num_workers
            )
            
            return train_dataloader, val_dataloader
            
        except Exception as e:
            self.custom_logger.error(f"Erro ao criar dataloaders: {str(e)}")
            raise
    
    def create_model(self, training: TimeSeriesDataSet) -> TemporalFusionTransformer:
        """Cria modelo TFT"""
        try:
            self.model = TemporalFusionTransformer.from_dataset(
                training,
                learning_rate=self.config.learning_rate,
                hidden_size=self.config.hidden_size,
                attention_head_size=self.config.attention_head_size,
                dropout=self.config.dropout,
                hidden_continuous_size=self.config.hidden_continuous_size,
                loss=self.metrics['quantile_loss'],
                reduce_on_plateau_patience=4,
                logging_metrics=self.metrics
            )
            
            return self.model
            
        except Exception as e:
            self.custom_logger.error(f"Erro ao criar modelo: {str(e)}")
            raise
            
    def train_model(self, df: pd.DataFrame) -> Dict:
        """Treina o modelo com validação cruzada temporal"""
        try:
            mlflow.set_experiment("market_prediction")
            
            with mlflow.start_run():
                # Prepara dados
                training, validation = self.create_datasets(df)
                train_dataloader, val_dataloader = self.create_dataloaders(training, validation)
                
                # Cria modelo
                self.model = self.create_model(training)
                
                # Configura callbacks
                callbacks = [
                    EarlyStopping(
                        monitor="val_loss",
                        min_delta=1e-4,
                        patience=5,
                        mode="min"
                    ),
                    ModelCheckpoint(
                        dirpath=self.config.MODEL_SAVE_PATH,
                        filename="tft-{epoch:02d}-{val_loss:.2f}",
                        monitor="val_loss",
                        mode="min"
                    )
                ]
                
                # Configura logger
                tb_logger = TensorBoardLogger(
                    save_dir=self.config.LOG_PATH,
                    name="tft_logs"
                )
                
                # Configura trainer
                trainer = pl.Trainer(
                    max_epochs=self.config.max_epochs,
                    accelerator="gpu" if torch.cuda.is_available() else "cpu",
                    devices=1,
                    callbacks=callbacks,
                    logger=tb_logger,
                    gradient_clip_val=0.1
                )
                
                # Treina modelo
                trainer.fit(
                    self.model,
                    train_dataloaders=train_dataloader,
                    val_dataloaders=val_dataloader
                )
                
                # Avalia modelo
                results = self.evaluate_model(val_dataloader)
                
                # Loga métricas no MLflow
                mlflow.log_metrics(results)
                
                # Salva modelo
                self.save_model()
                
                return results
                
        except Exception as e:
            self.custom_logger.error(f"Erro no treinamento: {str(e)}")
            raise

    def evaluate_model(self, dataloader: torch.utils.data.DataLoader) -> Dict[str, float]:
        """Avalia performance do modelo"""
        try:
            predictions = self.model.predict(dataloader)
            actuals = torch.cat([y for x, y in iter(dataloader)])
            
            results = {}
            for name, metric in self.metrics.items():
                results[name] = metric(predictions, actuals).item()
                
            return results
            
        except Exception as e:
            self.custom_logger.error(f"Erro na avaliação: {str(e)}")
            raise

    def predict(self, df: pd.DataFrame, n_samples: int = 100) -> Dict:
        """Realiza previsões com o modelo treinado"""
        try:
            if self.model is None:
                raise ValueError("Modelo não treinado")
                
            # Prepara dados para previsão
            dataset = TimeSeriesDataSet(
                df,
                time_idx="time_idx",
                target="ULTIMO_PRECO",
                group_ids=["group"],
                max_encoder_length=self.config.max_encoder_length,
                max_prediction_length=self.config.max_prediction_length,
                static_categoricals=[],
                static_reals=[],
                time_varying_known_reals=self.config.time_varying_known_reals,
                time_varying_unknown_reals=self.config.time_varying_unknown_reals,
                target_normalizer=GroupNormalizer(
                    groups=["group"],
                    transformation="softplus"
                ),
                add_relative_time_idx=True,
                add_target_scales=True,
                add_encoder_length=True,
            )
            
            dataloader = dataset.to_dataloader(
                train=False,
                batch_size=self.config.batch_size,
                num_workers=self.config.num_workers
            )
            
            # Realiza previsões
            raw_predictions = self.model.predict(
                dataloader,
                mode="raw",
                return_x=True,
                n_samples=n_samples
            )
            
            # Processa previsões
            predictions = {
                'forecast': raw_predictions.mean(dim=1),
                'lower_bound': raw_predictions.quantile(0.025, dim=1),
                'upper_bound': raw_predictions.quantile(0.975, dim=1),
                'uncertainty': (raw_predictions.quantile(0.975, dim=1) - 
                              raw_predictions.quantile(0.025, dim=1)) / 2
            }
            
            return predictions
            
        except Exception as e:
            self.custom_logger.error(f"Erro nas previsões: {str(e)}")
            raise

    def save_model(self, path: Optional[str] = None):
        try:
            if self.model is not None and self.model.model is not None:
                if path is None:
                    path = Path(self.config.MODEL_SAVE_PATH) / f"model_{datetime.now():%Y%m%d_%H%M}.pt"
                torch.save({
                    'model_state_dict': self.model.state_dict(),
                    'config': self.config,
                    'training_cutoff': self.training_cutoff
                }, path)
                self.custom_logger.info(f"Modelo salvo em {path}")
            else:
                self.custom_logger.warning("O modelo não pode ser salvo porque não foi treinado.")
        except Exception as e:
            self.custom_logger.error(f"Erro ao salvar o modelo: {str(e)}")
            raise

    def load_model(self, path: str):
        if not Path(path).exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {path}")
            
        checkpoint = torch.load(path)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.training_cutoff = checkpoint['training_cutoff']
        self.model.eval()
        
        self.custom_logger.info(f"Modelo carregado de {path}")
        
    def forward(self, x):
        """Forward pass"""
        return self.model(x)

    def training_step(self, batch, batch_idx):
        """Passo de treinamento"""
        y_pred = self(batch)
        loss = self.model.loss(batch, y_pred)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        """Passo de validação"""
        y_pred = self(batch)
        loss = self.model.loss(batch, y_pred)
        self.log("val_loss", loss)
        return loss

    def configure_optimizers(self):
        """Configura otimizador"""
        return torch.optim.Adam(self.parameters(), lr=self.config.learning_rate)

In [5]:
import logging
from typing import List, Dict, Optional
from datetime import datetime
import os
from pathlib import Path
import pdfkit

class MarketReport:
    def __init__(self, config: Config):
        self.config = config
        self.setup_logging()
        self.separator = "=" * 80
        self.template = self._create_template()
        
        self.template_required_fields = [
            'trend', 'trend_strength', 'volatility', 'last_price', 'vwap',
            'avg_volume', 'buy_sell_ratio', 'rsi', 'stoch_k', 'stoch_d',
            'poc_d', 'vah_d', 'val_d', 'poc_w', 'vah_w', 'val_w',
            'poc_m', 'vah_m', 'val_m', 'bb_upper', 'bb_lower',
            'next_resistance', 'next_support', 'resistance_distance', 'support_distance',
            'bullish_signals', 'bearish_signals', 'stoch_signal', 'rsi_signal', 'ma_signal',
            'vwap_position', 'vwap_distance'
        ]
        
        self.template_default_values = {
            'trend': 'LATERAL →',
            'trend_strength': 'MODERADA',
            'volatility': 0.0,
            'vwap_position': 'NA',
            'bullish_signals': 'Não',
            'bearish_signals': 'Não',
            'stoch_signal': 'NEUTRO',
            'rsi_signal': 'NEUTRO',
            'ma_signal': 'NEUTRO',
        }

    def setup_logging(self):
        """Configura sistema de logging"""
        log_file = Path(self.config.LOG_PATH) / f"market_report_{datetime.now():%Y%m%d}.log"
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)

    def format_level_detail(self, level: Dict) -> str:
        """Formata detalhes de um nível incluindo cálculo de distância"""
        strength_bar = self._create_strength_bar(level['strength'])
        volume_bar = self._create_volume_bar(level['volume'])
        
        last_price = level.get('last_price', 0)
        if last_price:
            distance = ((level['price'] - last_price) / last_price) * 100
        else:
            distance = 0.0
        
        return (
            f"Preço: {level['price']:>10,.2f} | "
            f"Volume: {level['volume']:>10,.0f} {volume_bar} | "
            f"Força: {strength_bar} | "
            f"Toques: {level.get('touches', 0):>2d} | "
            f"Buy Ratio: {level.get('buy_ratio', 0)*100:>5.1f}% | "
            f"Dist: {distance:>6.2f}% | "
            f"Último Teste: {level.get('last_touch', 'N/A')} | "
            f"Gap: {level.get('gap', 0)*100:>4.1f}%"
        )

    def format_levels(self, levels: List[Dict]) -> str:
        """Formata lista de níveis para o relatório"""
        if not levels:
            return "Nenhum nível identificado"
        
        formatted_levels = []
        for i, level in enumerate(levels, 1):
            prefix = f"{i:2d}."
            touches = level.get('touches', 0)
            strength_bar = self._create_strength_bar(level['strength'])
            details = (
                f"Preço: {level['price']:.2f}, "
                f"Volume: {level['volume']:.0f}, "
                f"Força: {strength_bar} ({level['strength']:.2f}), "
                f"Toques: {touches}"
            )
            formatted_levels.append(f"{prefix} {details}")
        
        return "\n".join(formatted_levels)
    
    def _create_strength_bar(self, strength: float, length: int = 10) -> str:
        """Cria barra visual de força"""
        filled = int(strength * length)
        return f"[{'■' * filled}{'□' * (length - filled)}]"

    def _create_volume_bar(self, volume: float, max_volume: Optional[float] = None, 
                          length: int = 5) -> str:
        """Cria barra visual de volume"""
        if max_volume is None:
            max_volume = volume * 2
        if max_volume == 0:
            return '[' + '░' * length + ']'
        filled = int((volume / max_volume) * length)
        return f"[{'▉' * filled}{'░' * (length - filled)}]"
    
    def _format_buy_sell_analysis(self, df: pd.DataFrame) -> str:
        """Formata análise de força compradora/vendedora"""
        recent = df.tail(5)
        buy_vol = recent['VOLUME_COMPRA'].sum()
        sell_vol = recent['VOLUME_VENDA'].sum()
        ratio = buy_vol / (buy_vol + sell_vol)
        
        return (
            f"Volume de Compra: {buy_vol:,.0f}\n"
            f"Volume de Venda: {sell_vol:,.0f}\n"
            f"Ratio Compra/Venda: {ratio:.2f}\n"
            f"Pressão: {'Compradora ↑' if ratio > 0.5 else 'Vendedora ↓'}"
        )

    def _format_gaps(self, df: pd.DataFrame) -> str:
        """Formata informações sobre gaps"""
        gaps = []
        for i in range(1, len(df)):
            prev_close = df['ULTIMO_PRECO'].iloc[i-1]
            curr_open = df['PRECO_ABERTURA'].iloc[i]
            gap_size = (curr_open - prev_close) / prev_close * 100
            
            if abs(gap_size) >= self.config.MAX_GAP_SIZE:
                gaps.append(
                    f"Gap {'de Alta' if gap_size > 0 else 'de Baixa'}: "
                    f"{gap_size:.2f}% em {df['DATA'].iloc[i]:%Y-%m-%d}"
                )
        
        return "\n".join(gaps) if gaps else "Nenhum gap significativo identificado"

    def _format_confluence_zones(self, supports: List[Dict], 
                               resistances: List[Dict]) -> str:
        """Formata zonas de confluência"""
        zones = []
        all_levels = supports + resistances
        
        for i, level1 in enumerate(all_levels):
            confluent_levels = []
            for j, level2 in enumerate(all_levels):
                if i != j:
                    price_diff = abs(level1['price'] - level2['price']) / level1['price']
                    if price_diff < self.config.CONFLUENCE_THRESHOLD:
                        confluent_levels.append(level2)
            
            if confluent_levels:
                zones.append(
                    f"Zona {level1['price']:.2f}: "
                    f"{len(confluent_levels)} níveis confluentes"
                )
        
        return "\n".join(zones) if zones else "Nenhuma zona de confluência significativa"

    def _format_volume_distribution(self, df: pd.DataFrame) -> str:
        """Formata distribuição de volume"""
        volume_at_price = df.groupby('ULTIMO_PRECO')['VOLUME'].sum()
        total_volume = volume_at_price.sum()
        
        return "\n".join(
            f"Preço {price:.2f}: {volume:,.0f} ({volume/total_volume*100:.1f}%)"
            for price, volume in volume_at_price.nlargest(5).items()
        )

    def _format_model_metrics(self, metrics: Dict) -> str:
        """Formata métricas do modelo"""
        if not metrics:
            return "Métricas não disponíveis"
            
        return "\n".join(f"{name}: {value:.4f}" for name, value in metrics.items())


    def generate_combined_levels_report(self, combined_supports: List[Dict], 
                                      combined_resistances: List[Dict], timestamp: str) -> str:
        """Gera relatório específico para níveis combinados"""
        report_sections = []
        
        # Cabeçalho
        report_sections.extend([
            f"{self.separator}",
            "NÍVEIS COMBINADOS DE TODOS OS TIMEFRAMES - WDO",
            f"Data/Hora: {timestamp}",
            f"{self.separator}\n",
            
            "TOP SUPORTES COMBINADOS:",
            f"{self.separator}",
            self.format_levels(combined_supports),
            
            f"\n{self.separator}\n",
            "TOP RESISTÊNCIAS COMBINADAS:",
            f"{self.separator}",
            self.format_levels(combined_resistances),
            
            f"\n{self.separator}\n",
            "ZONAS DE CONFLUÊNCIA:",
            f"{self.separator}",
            self._format_confluence_zones(combined_supports, combined_resistances),
            
            f"\n{self.separator}\n",
            "ANÁLISE DE FORÇA DOS NÍVEIS:",
            f"{self.separator}"
        ])
        
        # Adiciona análise de força para cada nível
        for i, support in enumerate(combined_supports, 1):
            strength_bar = self._create_strength_bar(support['strength'])
            report_sections.append(
                f"Suporte {i}: {support['price']:.2f} | "
                f"Força: {strength_bar} ({support['strength']:.2f}) | "
                f"Toques: {support.get('touches', 0)} | "
                f"Volume: {support['volume']:,.0f}"
            )
        
        report_sections.append("\n")
        
        for i, resistance in enumerate(combined_resistances, 1):
            strength_bar = self._create_strength_bar(resistance['strength'])
            report_sections.append(
                f"Resistência {i}: {resistance['price']:.2f} | "
                f"Força: {strength_bar} ({resistance['strength']:.2f}) | "
                f"Toques: {resistance.get('touches', 0)} | "
                f"Volume: {resistance['volume']:,.0f}"
            )
        
        # Rodapé
        report_sections.extend([
            f"\n{self.separator}",
            f"Relatório gerado em: {timestamp}",
            "Nota: Níveis combinados representam a agregação de todos os timeframes analisados",
            f"{self.separator}"
        ])
        
        return "\n".join(report_sections)
    

    def generate_consolidated_report(self, analyses: Dict, combined_supports: List[Dict],
                                   combined_resistances: List[Dict], predictions: Optional[Dict]) -> Dict[str, str]:
        try:
            current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            primary_sections = [
                f"{self.separator}",
                "ANÁLISE DE MERCADO CONSOLIDADA - WDO",
                f"Data/Hora: {current_time}",
                f"{self.separator}\n"
            ]
    
            for timeframe, analysis in analyses.items():
                context = analysis['context']
                df = analysis['df']
                
                primary_sections.extend([
                    f"ANÁLISE - TIMEFRAME {timeframe}",
                    f"{self.separator}",
                    "CONTEXTO DE MERCADO:",
                    f"Tendência: {context['trend']}",
                    f"Força da Tendência: {context['trend_strength']}",
                    f"Volatilidade: {context['volatility']:.2f}%",
                    f"Volume Médio: {context['avg_volume']:,.0f}",
                    f"RSI: {context['rsi']:.1f}",
                    f"Estocástico K/D: {context['stoch_k']:.1f}/{context['stoch_d']:.1f}\n",
                    "NÍVEIS DE SUPORTE:",
                    self.format_levels(analysis['supports']),
                    "\nNÍVEIS DE RESISTÊNCIA:",
                    self.format_levels(analysis['resistances']),
                    "\nANÁLISE DE VOLUME:",
                    self._format_volume_distribution(df),
                    "\nGAPS IMPORTANTES:",
                    self._format_gaps(df),
                    "\nFORÇA COMPRADORA/VENDEDORA:",
                    self._format_buy_sell_analysis(df),
                    f"\n{self.separator}\n"
                ])
    
            if predictions:
                primary_sections.extend([
                    "PREVISÕES DO MODELO",
                    f"{self.separator}",
                    f"Preço Previsto: {float(predictions['forecast'].mean()):.2f}",
                    f"Intervalo de Confiança: {float(predictions['lower_bound'].mean()):.2f} - "
                    f"{float(predictions['upper_bound'].mean()):.2f}",
                    f"Incerteza: ±{float(predictions['uncertainty'].mean()):.2f}",
                    f"\n{self.separator}\n"
                ])
    
            levels_sections = [
                "ZONAS DE CONFLUÊNCIA - NIVEIS COMBINADOS DE TODOS OS TIMEFRAMES",
                f"Data/Hora: {current_time}",
                f"{self.separator}",
                self._format_confluence_zones(combined_supports, combined_resistances),
                f"{self.separator}",
                "ANÁLISE DE FORÇA DOS NÍVEIS:",
            ]
    
            total_buy_volume = sum(df['VOLUME_COMPRA'].sum() for analysis in analyses.values() for df in [analysis['df']])
            total_sell_volume = sum(df['VOLUME_VENDA'].sum() for analysis in analyses.values() for df in [analysis['df']])
            overall_dominance = "DOMINIO COMPRADOR" if total_buy_volume > total_sell_volume else "DOMINIO VENDEDOR"
            
            levels_sections.append(f"Dominância geral: {overall_dominance}\n")
            
            for i, support in enumerate(combined_supports, 1):
                strength_bar = self._create_strength_bar(support['strength'])
                price = support['price']
                
                buy_volume = sum(df[df['ULTIMO_PRECO'] == price]['VOLUME_COMPRA'].sum() for analysis in analyses.values() for df in [analysis['df']])
                sell_volume = sum(df[df['ULTIMO_PRECO'] == price]['VOLUME_VENDA'].sum() for analysis in analyses.values() for df in [analysis['df']])
                
                dominance = "DOMINIO COMPRADOR" if buy_volume > sell_volume else "DOMINIO VENDEDOR"
                levels_sections.append(
                    f"Suporte {i}: {price:.2f} | "
                    f"Força: {strength_bar} ({support['strength']:.2f}) | "
                    f"Toques: {support.get('touches', 0)} | "
                    f"Volume: {support['volume']:,.0f} | "
                    f"Buy Volume: {buy_volume:,.0f} | "
                    f"Sell Volume: {sell_volume:,.0f} | "
                    f"{dominance}"
                )
    
            levels_sections.append("\n")
    
            for i, resistance in enumerate(combined_resistances, 1):
                strength_bar = self._create_strength_bar(resistance['strength'])
                price = resistance['price']
                
                buy_volume = sum(df[df['ULTIMO_PRECO'] == price]['VOLUME_COMPRA'].sum() for analysis in analyses.values() for df in [analysis['df']])
                sell_volume = sum(df[df['ULTIMO_PRECO'] == price]['VOLUME_VENDA'].sum() for analysis in analyses.values() for df in [analysis['df']])
                
                dominance = "DOMINIO COMPRADOR" if buy_volume > sell_volume else "DOMINIO VENDEDOR"
                levels_sections.append(
                    f"Resistência {i}: {price:.2f} | "
                    f"Força: {strength_bar} ({resistance['strength']:.2f}) | "
                    f"Toques: {resistance.get('touches', 0)} | "
                    f"Volume: {resistance['volume']:,.0f} | "
                    f"Buy Volume: {buy_volume:,.0f} | "
                    f"Sell Volume: {sell_volume:,.0f} | "
                    f"{dominance}"
                )
    
   
            return {
                'primary_report': "\n".join(primary_sections),
                'levels_report': "\n".join(levels_sections)
            }
    
        except Exception as e:
            self.logger.error(f"Erro ao gerar relatório consolidado: {str(e)}")
            raise
    

    def _convert_to_html(self, report: str) -> str:
        """Converte relatório para HTML com formatação"""
        css = """
    body {
        font-family: Consolas, Monaco, monospace;
        font-size: 12px;
        line-height: 1.4;
        white-space: pre-wrap;
        background-color: white;
        color: black;
        padding: 20px;
    }
    .separator {
        border-top: 1px solid #000;
        margin: 10px 0;
    }
    .level {
        margin: 5px 0;
    }
    .strength-bar, .volume-bar {
        font-family: Consolas, Monaco, monospace;
        color: #000;
    }
    table {
        width: 100%;
        border-collapse: collapse;
    }
    th, td {
        border: 1px solid #ddd;
        padding: 8px;
        text-align: left;
    }
    th {
        background-color: #f5f5f5;
    }
    """
        
        html_template = """<!DOCTYPE html>
    <html>
    <head>
    <meta charset="UTF-8">
    <style>
    {}
    </style>
    </head>
    <body>
    {}
    </body>
    </html>
    """
    
        content = report.replace(self.separator, '<hr class="separator">')
        content = content.replace('\n', '<br>')
        return html_template.format(css, content)

    def save_report(self, report: Dict[str, str]) -> Dict[str, str]:
        """Salva relatório em múltiplos formatos"""
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            saved_files = {}
                       
            # Generate PDFs
            pdf_path = self.export_to_pdf(report['primary_report'], "report")
            levels_pdf_path = self.export_to_pdf(report['levels_report'], "levels")
            
            if pdf_path:
                saved_files['primary_pdf'] = pdf_path
            if levels_pdf_path:
                saved_files['levels_pdf'] = levels_pdf_path
            
            return saved_files
            
        except Exception as e:
            self.logger.error(f"Erro ao salvar relatório: {str(e)}")
            raise

    def export_to_pdf(self, report: str, prefix: str = "report") -> Optional[str]:
        """Exporta relatório para PDF"""
        try:
            pdf_dir = Path(self.config.CACHE_DIR) / "pdf"
            pdf_dir.mkdir(exist_ok=True)
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            pdf_path = pdf_dir / f"{prefix}_{timestamp}.pdf"
            
            html_content = self._convert_to_html(report)
            
            options = {
                'page-size': 'A4',
                'margin-top': '0.75in',
                'margin-right': '0.75in',
                'margin-bottom': '0.75in',
                'margin-left': '0.75in',
                'encoding': 'UTF-8',
                'no-outline': None
            }
            
            pdfkit.from_string(html_content, str(pdf_path), options=options, 
                             configuration=pdfkit.configuration(wkhtmltopdf='C:\\Program Files\\wkhtmltopdf\\bin\\wkhtmltopdf.exe'))
            self.logger.info(f"PDF gerado em {pdf_path}")
            return str(pdf_path)
            
        except Exception as e:
            self.logger.error(f"Erro ao exportar PDF: {str(e)}")
            return None

    def _create_template(self) -> str:
            """Cria template do relatório"""
            return """
    {separator}
    ANÁLISE DE MERCADO - WDO - Multi-Timeframe
    {separator}
    Data/Hora: {datetime:%Y-%m-%d %H:%M:%S %Z}
    Último Fechamento: {last_close:.2f}
    
    {separator}
    RESUMO DO MERCADO
    {separator}
    Tendência: {trend}
    Força da Tendência: {trend_strength}
    Volatilidade: {volatility:.2f}%
    Volume Médio (5 períodos): {avg_volume:,.0f}
    Volume Buy/Sell Ratio: {buy_sell_ratio:.2f}
    RSI Atual: {rsi:.1f}
    VWAP: {vwap:.2f}
    Estocástico (K/D): {stoch_k:.1f}/{stoch_d:.1f}
    
    {separator}
    NÍVEIS DE REFERÊNCIA
    {separator}
    Point of Control Diário: {poc_d:.2f}
    Value Area High Diário: {vah_d:.2f}
    Value Area Low Diário: {val_d:.2f}
    
    Point of Control Semanal: {poc_w:.2f}
    Value Area High Semanal: {vah_w:.2f}
    Value Area Low Semanal: {val_w:.2f}
    
    Point of Control Mensal: {poc_m:.2f}
    Value Area High Mensal: {vah_m:.2f}
    Value Area Low Mensal: {val_m:.2f}
    
    {separator}
    TOP NÍVEIS DE RESISTÊNCIA (Ordenados por Volume e Força)
    {separator}
    {resistance_levels}
    
    {separator}
    TOP NÍVEIS DE SUPORTE (Ordenados por Volume e Força)
    {separator}
    {support_levels}
    
    {separator}
    ANÁLISE ADICIONAL
    {separator}
    Gaps Importantes:
    {gaps}
    
    Zonas de Confluência:
    {confluence_zones}
    
    Próximos Níveis Críticos:
    ↑ Resistência: {next_resistance:.2f} (distância: {resistance_distance:.2f}%)
    ↓ Suporte: {next_support:.2f} (distância: {support_distance:.2f}%)
    
    {separator}
    ANÁLISE DE VOLUME E LIQUIDEZ
    {separator}
    Distribuição de Volume por Nível:
    {volume_distribution}
    
    Força Compradora/Vendedora:
    {buy_sell_analysis}
    
    {separator}
    SINAIS TÉCNICOS
    {separator}
    Regular Bullish: {bullish_signals}
    Regular Bearish: {bearish_signals}
    Estocástico: {stoch_signal}
    RSI: {rsi_signal}
    EMA vs SMA: {ma_signal}
    
    {separator}
    ANÁLISE VWAP
    {separator}
    Posição em relação ao VWAP: {vwap_position}
    Bandas de Bollinger:
    Superior: {bb_upper:.2f}
    Inferior: {bb_lower:.2f}
    Distância do VWAP: {vwap_distance:.2f}%
    
    {separator}
    PREVISÕES DO MODELO
    {separator}
    Previsão de Preço (próximas 24 horas):
    Preço Previsto: {forecast_price:.2f}
    Intervalo de Confiança: {forecast_lower:.2f} - {forecast_upper:.2f}
    Incerteza: ±{forecast_uncertainty:.2f}
    Métricas do Modelo:
    {model_metrics}
    
    {separator}
    Gerado por Market Analyzer v3.0 (TradingView Edition)
    Nota: Níveis são baseados em análise histórica e devem ser usados em conjunto com outras ferramentas.
    {separator}
    """

In [6]:
import os
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple
import pandas as pd
import mlflow
import torch

class MarketAnalysisSystem:
    def __init__(self, config: Config):
        self.config = config
        self.setup_logging()
        self.data_processor = DataProcessor(config)
        self.market_analyzer = MarketAnalyzer(config)
        self.model = TFTModel(config)
        self.report_generator = MarketReport(config)  
    
    def setup_logging(self):
        log_file = Path(self.config.LOG_PATH) / f"system_{datetime.now():%Y%m%d}.log"
        logging.basicConfig(
            level=logging.WARNING,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_file),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
        
    def check_environment(self) -> bool:
        """Verifica ambiente de execução"""
        try:
            # Verifica diretórios
            if not os.path.exists(self.config.BASE_DIR):
                raise ValueError(f"Diretório base não encontrado: {self.config.BASE_DIR}")
            
            # Verifica arquivos de dados
            files = [f for f in os.listdir(self.config.BASE_DIR) 
                    if f.startswith('BMFBOVESPA_') and f.endswith('.csv')]
            if not files:
                raise ValueError("Nenhum arquivo de dados encontrado")
            
            # Verifica GPU
            if torch.cuda.is_available():
                self.logger.info(f"GPU disponível: {torch.cuda.get_device_name(0)}")
            else:
                self.logger.warning("GPU não disponível, usando CPU")
            
            self.logger.info("Verificação de ambiente concluída com sucesso")
            return True
            
        except Exception as e:
            self.logger.error(f"Erro na verificação do ambiente: {str(e)}")
            return False
            
    def process_timeframe(self, file: str) -> Dict:
        """Processa um timeframe específico com tratamento de erros melhorado"""
        try:
            # Extrai timeframe do nome do arquivo
            timeframe = file.split('_')[2].split('.')[0]
            filepath = Path(self.config.BASE_DIR) / file
            
            self.logger.info(f"Processando timeframe {timeframe}")
            
            # Carrega e processa dados com validação
            df = self.data_processor.load_and_process_data(filepath)
            if df is None or df.empty:
                raise ValueError(f"Dados inválidos para o timeframe {timeframe}")
                
            # Adiciona último preço ao contexto para cálculo de distância
            last_price = df['ULTIMO_PRECO'].iloc[-1]
            
            # Realiza análise de mercado
            supports, resistances = self.market_analyzer.identify_levels(df)
            context = self.market_analyzer.analyze_market_context(df)
            
            # Adiciona último preço aos níveis para cálculo de distância
            for level in supports + resistances:
                level['last_price'] = last_price
            
            return {
                'timeframe': timeframe,
                'df': df,
                'supports': supports,
                'resistances': resistances,
                'context': context
            }
                
        except Exception as e:
            self.logger.error(f"Erro ao processar timeframe {timeframe}: {str(e)}")
            raise

            
    def train_and_predict(self, df: pd.DataFrame) -> Optional[Dict]:
        try:
            self.logger.info("Iniciando treinamento do modelo")
            
            # Prepara dados para treino
            train_df, val_df = self.data_processor.prepare_train_val_data(df)
            
            # Treina modelo
            train_result = self.model.train_model(train_df)
            
            if train_result is not None:
                # Salva modelo
                self.model.save_model()
                
                # Realiza previsões
                predictions = self.model.predict(df)
                return predictions
            else:
                self.logger.warning("Não foi possível realizar previsões porque o modelo não foi treinado corretamente.")
                return None
            
        except Exception as e:
            self.logger.error(f"Erro no treinamento/previsão: {str(e)}")
            return None
            
    def combine_levels(self, analyses: Dict[str, Dict]) -> Tuple[List[Dict], List[Dict]]:
        """Combina níveis de diferentes timeframes"""
        all_supports = []
        all_resistances = []
        
        for analysis in analyses.values():
            all_supports.extend(analysis['supports'])
            all_resistances.extend(analysis['resistances'])
        
        # Ordena por força
        all_supports = sorted(all_supports, key=lambda x: x['strength'], reverse=True)
        all_resistances = sorted(all_resistances, key=lambda x: x['strength'], reverse=True)
        
        return all_supports[:10], all_resistances[:10]


    def run_analysis(self):
        try:
            self.logger.info("Iniciando análise de mercado")
                  
            # Verifica ambiente
            if not self.check_environment():
                raise EnvironmentError("Ambiente não configurado corretamente")
    
            # Lista arquivos
            files = [
                f for f in os.listdir(self.config.BASE_DIR)
                if f.startswith('BMFBOVESPA_') and f.endswith('.csv')
            ]
    
            # Processa cada timeframe
            analyses = {}
            for file in files:
                analysis = self.process_timeframe(file)
                if analysis['timeframe'] == 'D':  # Assumindo que queremos usar o TFT apenas para o timeframe diário
                    supports, resistances = self.market_analyzer.identify_levels_with_tft(analysis['df'], self.model)
                else:
                    supports, resistances = self.market_analyzer.identify_levels(analysis['df'])
                analysis['supports'] = supports
                analysis['resistances'] = resistances
                analyses[analysis['timeframe']] = analysis
    
                           
            # Combina níveis
            combined_supports, combined_resistances = self.combine_levels(analyses)
    
            # Treina modelo e faz previsões
            predictions = None
            if 'D' in analyses:  # Usa timeframe diário para previsões
                predictions = self.train_and_predict(analyses['D']['df'])
                if predictions is None:
                    self.logger.warning("Não foi possível realizar previsões porque o modelo não foi treinado corretamente.")
    
            # Gera relatórios separados
            reports = self.report_generator.generate_consolidated_report(
                analyses, combined_supports, combined_resistances, predictions
            )
    
            # Salva ambos os relatórios
            report_files = self.report_generator.save_report(reports)
    
            self.logger.info("Análise concluída com sucesso")
    
            # Salva resultados
            self._save_results(analyses, reports, predictions)
    
            return {
                'analyses': analyses,
                'predictions': predictions,
                'reports': reports,
                'report_files': report_files
            }
    
        except Exception as e:
            self.logger.error(f"Erro durante análise: {str(e)}")
            raise

    
    def _save_results(self, analyses: Dict, reports: Dict[str, str], predictions: Optional[Dict]):
        """Salva resultados da análise"""
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            # Salva análises
            for timeframe, analysis in analyses.items():
                analysis['df'].to_parquet(
                    Path(self.config.CACHE_DIR) / 
                    f"analysis_{timeframe}_{timestamp}.parquet"
                )
            
            # Salva relatórios separadamente
            for report_type, content in reports.items():
                filename = "combined_levels" if report_type == "levels_report" else "analysis"
                report_path = Path(self.config.CACHE_DIR) / f"{filename}_{timestamp}.txt"
                with open(report_path, 'w', encoding='utf-8') as f:
                    f.write(content)
            
            # Salva previsões
            if predictions:
                pd.to_pickle(
                    predictions,
                    Path(self.config.CACHE_DIR) / f"predictions_{timestamp}.pkl"
                )
            
            self.logger.info(f"Resultados salvos com timestamp {timestamp}")
            
        except Exception as e:
            self.logger.error(f"Erro ao salvar resultados: {str(e)}")
            raise


def main():
    """Função principal"""
    try:
        # Configura MLflow
        mlflow.set_tracking_uri("sqlite:///mlflow.db")
        mlflow.set_experiment("market_analysis")
        
        # Inicializa sistema
        config = Config()
        system = MarketAnalysisSystem(config)
        
        # Executa análise
        results = system.run_analysis()
        
        print("\nAnálise concluída!")
        print(f"Relatório gerado em: {config.CACHE_DIR}")
        
        return results
        
    except Exception as e:
        print(f"\nErro durante execução: {str(e)}")
        return None

if __name__ == "__main__":
    main()

2024-11-22 20:12:53,083 - __main__.1439022002960 - INFO - GPU disponível: NVIDIA GeForce RTX 4070 Laptop GPU


2024-11-22 20:12:53,083 - __main__.1439022002960 - INFO - GPU disponível: NVIDIA GeForce RTX 4070 Laptop GPU



Análise concluída!
Relatório gerado em: C:/ARQUIVOS/OneDrive/Bolsa/COLETA/TV/FUTUROS/cache
